In [2]:
import gc

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import default_risk.config

application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application_ready_to_join.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")


merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

merged_df = merged_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)

del bureau_df, prev_app_df
gc.collect()

load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)

merged_df.head()


,id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,...,amt_goods_price_is_missing_sum,rate_down_payment_is_missing_mean,rate_down_payment_is_missing_sum,days_decision_mean,days_decision_min,days_decision_max,cnt_payment_mean,cnt_payment_min,cnt_payment_max,cnt_payment_sum
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,0.000000,0.0,-606.000000,-606.0,-606.0,24.000000,24.0,24.0,24.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.333333,1.0,-1305.000000,-2341.0,-746.0,10.000000,6.0,12.0,30.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.000000,0.0,-815.000000,-815.0,-815.0,4.000000,4.0,4.0,4.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,3.0,0.777778,7.0,-272.444444,-617.0,-181.0,23.000000,0.0,48.0,138.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.666667,4.0,-1222.833333,-2357.0,-374.0,20.666667,10.0,48.0,124.0


In [3]:
Y= merged_df["target"]
X= merged_df.drop(columns=["target"])
X.drop(columns=["id_curr"],inplace=True)

categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X[col] = X[col].astype('category')

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

hiperparams=  {  
    "objective" : 'binary:logistic',
    "random_state" : 42,
    "eval_metric" :"auc",
    "enable_categorical" : True
}

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline+bureau+prev")

[0]	validation_0-auc:0.71372
[1]	validation_0-auc:0.72447
[2]	validation_0-auc:0.72929
[3]	validation_0-auc:0.73221
[4]	validation_0-auc:0.73550
[5]	validation_0-auc:0.73839
[6]	validation_0-auc:0.74131
[7]	validation_0-auc:0.74404
[8]	validation_0-auc:0.74652
[9]	validation_0-auc:0.75023
[10]	validation_0-auc:0.75145
[11]	validation_0-auc:0.75237
[12]	validation_0-auc:0.75317
[13]	validation_0-auc:0.75474
[14]	validation_0-auc:0.75542
[15]	validation_0-auc:0.75681
[16]	validation_0-auc:0.75727
[17]	validation_0-auc:0.75754
[18]	validation_0-auc:0.75837
[19]	validation_0-auc:0.75935
[20]	validation_0-auc:0.76034
[21]	validation_0-auc:0.76034
[22]	validation_0-auc:0.76118
[23]	validation_0-auc:0.76144
[24]	validation_0-auc:0.76141
[25]	validation_0-auc:0.76184
[26]	validation_0-auc:0.76203
[27]	validation_0-auc:0.76263
[28]	validation_0-auc:0.76285
[29]	validation_0-auc:0.76266
[30]	validation_0-auc:0.76304
[31]	validation_0-auc:0.76323
[32]	validation_0-auc:0.76351
[33]	validation_0-au